# Phase 5: SHAP explainability

This notebook starts fresh, so we recreate the same data, split, and
Random Forest model from Phase 4 (same random_state throughout), then
use SHAP to explain individual predictions.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, average_precision_score

df = pd.read_csv("data/raw/creditcard.csv")

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

y_scores_rf = model_rf.predict_proba(X_test)[:, 1]
y_pred_rf = (y_scores_rf >= 0.40).astype(int)  # chosen threshold from Phase 4

precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
auprc_rf = average_precision_score(y_test, y_scores_rf)

print(f"Precision: {precision_rf:.4f}  (Phase 4: 0.9333)")
print(f"Recall:    {recall_rf:.4f}  (Phase 4: 0.8571)")
print(f"AUPRC:     {auprc_rf:.4f}  (Phase 4: 0.8734)")

Precision: 0.9333  (should match Phase 4: 0.9333)
Recall:    0.8571  (should match Phase 4: 0.8571)
AUPRC:     0.8734  (should match Phase 4: 0.8734)


## Setting up SHAP

TreeExplainer is a fast, exact SHAP method built specifically for
tree-based models like Random Forest, it uses the actual tree
structure rather than approximating.

We'll start by explaining a single transaction's prediction, to see
concretely what a SHAP value represents, before scaling up.

In [2]:
import shap
import numpy as np

explainer = shap.TreeExplainer(model_rf)

# pick one transaction to explain, e.g. the first row of the test set
sample = X_test.iloc[[0]]
shap_values = explainer.shap_values(sample)

print("Shape of shap_values:", np.array(shap_values).shape)

Shape of shap_values: (1, 30, 2)


In [3]:
# fraud class is index 1
fraud_shap_values = shap_values[0, :, 1]

# pair each SHAP value with its feature name, sorted by impact
contributions = pd.Series(fraud_shap_values, index=X_test.columns).sort_values(key=abs, ascending=False)

print("Actual class:", y_test.iloc[0])
print("Predicted probability of fraud:", y_scores_rf[0])
print("\nTop contributing features:")
print(contributions.head(10))

Actual class: 0
Predicted probability of fraud: 0.0

Top contributing features:
V14   -0.000473
V17   -0.000410
V12   -0.000290
V10   -0.000205
V4    -0.000147
V5    -0.000122
V11   -0.000116
V27    0.000113
V8     0.000103
V16   -0.000086
dtype: float64


In [4]:
# find positions (not the original dataframe index) where the model correctly caught fraud
true_positive_positions = np.where((y_test.values == 1) & (y_pred_rf == 1))[0]

print(f"Number of true positives found: {len(true_positive_positions)}")
print("First few positions:", true_positive_positions[:5])

Number of true positives found: 84
First few positions: [ 840 1146 3287 4276 5077]


In [5]:
tp_position = true_positive_positions[0]

sample_tp = X_test.iloc[[tp_position]]
shap_values_tp = explainer.shap_values(sample_tp)

fraud_shap_values_tp = shap_values_tp[0, :, 1]
contributions_tp = pd.Series(fraud_shap_values_tp, index=X_test.columns).sort_values(key=abs, ascending=False)

print("Actual class:", y_test.iloc[tp_position])
print("Predicted probability of fraud:", y_scores_rf[tp_position])
print("\nTop contributing features:")
print(contributions_tp.head(10))

Actual class: 1
Predicted probability of fraud: 0.9

Top contributing features:
V17    0.310499
V14    0.192532
V12    0.119007
V10    0.091511
V16    0.090252
V7     0.018613
V26    0.014565
V21    0.013144
V4     0.008397
V9     0.008056
dtype: float64


## Global feature importance across the test set

Rather than explaining one transaction, we compute SHAP values across
a sample of the test set and average their absolute values per
feature. This reveals which features the model relies on most overall,
not just for one prediction, confirming (or challenging) what we saw
with V17 and V14 in the single-transaction examples.

In [6]:
# sample for speed, keeping it reasonably large but manageable
sample_size = 2000
X_sample = X_test.sample(n=sample_size, random_state=42)

shap_values_sample = explainer.shap_values(X_sample)

# fraud class is index 1, same as before
fraud_shap_sample = shap_values_sample[:, :, 1]

# mean absolute SHAP value per feature, across all sampled transactions
mean_abs_shap = pd.Series(
    np.abs(fraud_shap_sample).mean(axis=0), index=X_test.columns
).sort_values(ascending=False)

print("Top 10 features by average impact on fraud predictions:")
print(mean_abs_shap.head(10))

Top 10 features by average impact on fraud predictions:
V17    0.000862
V14    0.000792
V12    0.000633
V1     0.000444
V4     0.000423
V10    0.000353
V28    0.000267
V16    0.000231
V11    0.000211
V2     0.000175
dtype: float64
